# 14. 정규화 기법 — 과대적합과 싸우기

> **제14장** · **이론편 대응: 11.3절(정규화), 11.6절(조기 종료)**
> **예상 소요**: 80분
> **필요 사양**: **[CPU]** 로 실행 가능
> **추가 설치**: 없음
> **다운로드**: FashionMNIST (13장에서 받았다면 재사용)

---

## 이 장에서 하는 일

9장에서 결정트리가 학습 데이터를 100% 맞히면서 시험에서 무너지는 것을 봤다.
**신경망도 같은 문제를 겪는다.** 오히려 더 심하다.

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 1 | 과대적합을 만들어 보기 | 8.5절 |
| 2 | **가중치 감쇠 (L2)** | 11.3절 |
| 3 | **Dropout — 손계산 검증** ★ | 11.3절 |
| 4 | **Batch Normalization 손계산** ★ | 11.3절 |
| 5 | 실제 학습에서 비교 | 11.3절 |
| 6 | **조기 종료** | 11.6절 |
| 7 | 데이터 증강 | 11.4절 |
| 8 | 무엇을 언제 쓸 것인가 | 11.3절 |

**5절은 정직하게 다룬다.** 정규화가 항상 극적인 효과를 내지는 않는다는 것,
그리고 **무엇을 기준으로 봐야 하는지**를 확인한다.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform
from pathlib import Path

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

np.set_printoptions(precision=4, suppress=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} / 장치: {device}")

---

## 1. 과대적합을 만들어 보기 — 이론편 8.5절

**과대적합이 잘 일어나는 조건을 일부러 만든다.**

| 조건 | 설정 |
|---|---|
| 데이터가 적음 | 학습 500장만 사용 |
| 모델이 큼 | 은닉층 512x2 (파라미터 약 67만 개) |
| 오래 학습 | 150에폭 |

**파라미터가 데이터보다 훨씬 많다.** 외우기 딱 좋은 조건이다.

In [ ]:
import torch
from torchvision import datasets, transforms
from pathlib import Path

root = Path.cwd()
if root.name.startswith("part"):
    root = root.parent

print("데이터 준비 중... (13장에서 받았다면 즉시)")

train_data = datasets.FashionMNIST(
    root=str(root / "data"), train=True, download=True,
    transform=transforms.ToTensor())
test_data = datasets.FashionMNIST(
    root=str(root / "data"), train=False, download=True,
    transform=transforms.ToTensor())

N_TRAIN, N_TEST = 500, 1500

X_train = torch.stack([train_data[i][0] for i in range(N_TRAIN)]).view(N_TRAIN, -1)
y_train = torch.tensor([train_data[i][1] for i in range(N_TRAIN)])
X_test = torch.stack([test_data[i][0] for i in range(N_TEST)]).view(N_TEST, -1)
y_test = torch.tensor([test_data[i][1] for i in range(N_TEST)])

print()
print("=" * 70)
print("실습 설정")
print("=" * 70)
print(f"  학습 데이터: {N_TRAIN}장  (일부러 적게)")
print(f"  시험 데이터: {N_TEST}장")
print(f"  입력 차원  : {X_train.shape[1]}")
print(f"  클래스     : 10개")
print()

# 모델 파라미터 수 계산
n_params = 784*512 + 512 + 512*512 + 512 + 512*10 + 10
print(f"  모델 파라미터: 약 {n_params:,}개")
print(f"  학습 표본 수 : {N_TRAIN}개")
print(f"  비율        : 파라미터가 표본의 {n_params/N_TRAIN:,.0f}배")
print()
print("이 조건이면 모델이 학습 데이터를 통째로 외울 수 있다.")

In [ ]:
import torch
import torch.nn as nn
import numpy as np


def make_model(dropout=0.0, batchnorm=False, hidden=512, seed=42):
    """실습용 MLP — 정규화 옵션을 켜고 끌 수 있다"""
    torch.manual_seed(seed)
    layers = [nn.Linear(784, hidden)]
    if batchnorm:
        # ── nn.BatchNorm1d 파라미터 ──────────────────────────────────
        #   num_features  정규화할 특성 수.  **필수** (앞 층의 출력 차원)
        #   eps           0 나눗셈 방지.  기본값 1e-5
        #   momentum      이동평균 갱신 비율.  기본값 0.1
        #                 running_mean = (1-momentum)*old + momentum*batch
        #   affine        학습 가능한 gamma·beta 사용.  기본값 True
        #                 False 면 순수 정규화만 한다
        #   track_running_stats  추론용 통계 누적.  기본값 True
        #
        #   [주의] 학습 시 배치 크기가 1이면 분산이 정의되지 않는다
        #          → DataLoader 에 drop_last=True 를 주거나 LayerNorm 사용
        # ──────────────────────────────────────────────────────────────
        layers.append(nn.BatchNorm1d(hidden))
    # ── nn.Dropout 파라미터 ──────────────────────────────────────
    #   p        끄는 비율.  기본값 0.5
    #            예: 0.1~0.3(가벼움) / 0.5(강함, FC 층)
    #            CNN 에서는 0.1~0.25 정도를 쓴다
    #   inplace  제자리 연산.  기본값 False
    #
    #   [중요] 학습 시에만 동작한다
    #     model.train()  → 뉴런을 끄고 남은 것에 1/(1-p) 를 곱함
    #     model.eval()   → 아무것도 하지 않음
    #     eval() 을 잊으면 예측이 매번 달라진다
    # ──────────────────────────────────────────────────────────────
    layers += [nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, hidden)]
    if batchnorm:
        layers.append(nn.BatchNorm1d(hidden))
    layers += [nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, 10)]
    return nn.Sequential(*layers)


def train_model(model, epochs=150, lr=1e-3, weight_decay=0.0,
                batch_size=32, verbose=False, seed=42):
    """학습하며 매 에폭 지표를 기록한다"""
    torch.manual_seed(seed)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr,
                                 weight_decay=weight_decay)
    # ── nn.CrossEntropyLoss 파라미터 ─────────────────────────────
    #   weight        클래스별 가중치.  기본값 None
    #                 불균형 데이터에서 소수 클래스에 큰 값을 준다
    #   ignore_index  무시할 라벨.  기본값 -100
    #                 **SFT 의 손실 마스킹이 이것을 이용한다**
    #   reduction     집계 방식.  기본값 'mean'
    #                 'mean'(평균) / 'sum'(합) / 'none'(개별 반환)
    #   label_smoothing  라벨 스무딩.  기본값 0.0
    #                    예: 0.1 — 과신을 줄여 일반화 향상
    #
    #   [주의] 입력은 **로짓**(소프트맥스 이전)이어야 한다
    #          내부에서 log_softmax 를 적용하므로 중복하면 안 된다
    # ──────────────────────────────────────────────────────────────
    criterion = nn.CrossEntropyLoss()

    Xtr, ytr = X_train.to(device), y_train.to(device)
    Xte, yte = X_test.to(device), y_test.to(device)

    history = {"train_loss": [], "test_loss": [],
               "train_acc": [], "test_acc": []}

    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), batch_size):
            idx = perm[i:i+batch_size]
            optimizer.zero_grad()
            criterion(model(Xtr[idx]), ytr[idx]).backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            out_tr, out_te = model(Xtr), model(Xte)
            history["train_loss"].append(criterion(out_tr, ytr).item())
            history["test_loss"].append(criterion(out_te, yte).item())
            history["train_acc"].append((out_tr.argmax(1) == ytr).float().mean().item())
            history["test_acc"].append((out_te.argmax(1) == yte).float().mean().item())

        if verbose and (epoch + 1) % 30 == 0:
            print(f"    에폭 {epoch+1:3}: 학습 {history['train_acc'][-1]:.4f} / "
                  f"시험 {history['test_acc'][-1]:.4f}")

    return history


print("=" * 70)
print("정규화 없이 학습 — 과대적합 재현")
print("=" * 70)

import time
t0 = time.time()
baseline = train_model(make_model(), epochs=150, verbose=True)
print(f"  소요 {time.time()-t0:.0f}초")
print()
print(f"  최종 학습 정확도: {baseline['train_acc'][-1]:.4f}")
print(f"  최종 시험 정확도: {baseline['test_acc'][-1]:.4f}")
print(f"  격차            : {baseline['train_acc'][-1] - baseline['test_acc'][-1]:+.4f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

epochs_range = range(1, len(baseline["train_acc"]) + 1)

# --- 왼쪽: 정확도 ---
ax = axes[0]
ax.plot(epochs_range, baseline["train_acc"], linewidth=2,
        color="#DC2626", label="학습 정확도")
ax.plot(epochs_range, baseline["test_acc"], linewidth=2,
        color="#0D9488", label="시험 정확도")
best_i = int(np.argmax(baseline["test_acc"]))
ax.axvline(best_i + 1, color="#1E40AF", linestyle="--", linewidth=1.5)
ax.text(best_i + 4, 0.4, f"시험 최고\n{best_i+1}에폭", fontsize=8, color="#1E40AF")
ax.set_xlabel("에폭")
ax.set_ylabel("정확도")
ax.set_title("정확도 곡선")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# --- 오른쪽: 손실 ---
ax = axes[1]
ax.plot(epochs_range, baseline["train_loss"], linewidth=2,
        color="#DC2626", label="학습 손실")
ax.plot(epochs_range, baseline["test_loss"], linewidth=2,
        color="#0D9488", label="시험 손실")
min_i = int(np.argmin(baseline["test_loss"]))
ax.axvline(min_i + 1, color="#1E40AF", linestyle="--", linewidth=1.5)
ax.text(min_i + 4, max(baseline["test_loss"]) * 0.75,
        f"시험 손실 최저\n{min_i+1}에폭", fontsize=8, color="#1E40AF")
ax.set_xlabel("에폭")
ax.set_ylabel("손실")
ax.set_title("손실 곡선")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("=" * 78)
print("두 그래프에서 읽을 것")
print("=" * 78)
print(f"  학습 정확도는 {baseline['train_acc'][-1]:.4f} 로 사실상 1.0")
print(f"  시험 정확도는 {baseline['test_acc'][-1]:.4f} 에서 더 오르지 않는다")
print()
print(f"  시험 **손실**은 {min_i+1}에폭에서 최저({min(baseline['test_loss']):.4f})를 찍고")
print(f"  이후 {baseline['test_loss'][-1]:.4f} 까지 다시 올라간다")
print()
print("[중요] 정확도보다 손실을 보라")
print("  정확도는 '맞았나 틀렸나'만 본다 — 확신이 잘못 커져도 안 보인다.")
print("  손실은 '얼마나 확신했나'까지 반영한다 — 과대적합이 먼저 드러난다.")

---

## 2. 가중치 감쇠 (L2) — 이론편 11.3절

**가중치가 커지는 것을 억제한다.**

$$L_{\text{전체}} = L_{\text{데이터}} + \lambda \sum_i w_i^2$$

큰 가중치에 벌점을 주므로, 모델이 **꼭 필요한 만큼만** 가중치를 키우게 된다.

**왜 가중치가 크면 문제인가**

가중치가 크면 입력의 작은 변화에도 출력이 크게 흔들린다.
학습 데이터의 미세한 특징(노이즈 포함)까지 붙잡게 되는 것이다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

print("=" * 78)
print("L2 강도에 따른 변화 — 로지스틱 회귀로 확인")
print("=" * 78)
print()
print("scikit-learn 의 C 는 정규화 강도의 **역수**다.")
print("  C 가 작을수록 강한 정규화")
print()

Xc, yc = make_classification(n_samples=300, n_features=50, n_informative=8,
                             n_redundant=5, random_state=42, flip_y=0.1)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
    Xc, yc, test_size=0.3, random_state=42, stratify=yc)
sc = StandardScaler().fit(Xc_tr)
Xc_tr_s, Xc_te_s = sc.transform(Xc_tr), sc.transform(Xc_te)

Cs = [100, 10, 1, 0.1, 0.05, 0.01]
l2_results = []

print(f"{'C':<10}{'정규화 강도':<16}{'학습':<12}{'시험':<12}{'가중치 노름'}")
print("-" * 78)
for C in Cs:
    m = LogisticRegression(C=C, max_iter=3000).fit(Xc_tr_s, yc_tr)
    tr, te = m.score(Xc_tr_s, yc_tr), m.score(Xc_te_s, yc_te)
    norm = float(np.linalg.norm(m.coef_))
    l2_results.append((C, tr, te, norm))
    strength = "약함" if C >= 10 else ("보통" if C >= 0.1 else "강함")
    print(f"{C:<10}{strength:<16}{tr:<12.4f}{te:<12.4f}{norm:.4f}")
print("-" * 78)

best = max(l2_results, key=lambda r: r[2])
print(f"시험 정확도 최고: C={best[0]} ({best[2]:.4f})")
print()
print("정규화를 강하게 하면")
print("  학습 정확도는 떨어진다 (덜 외운다)")
print("  가중치 노름이 작아진다")
print("  시험 정확도는 어느 지점까지 오르다가 다시 떨어진다")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

Cs_arr = [r[0] for r in l2_results]
trs = [r[1] for r in l2_results]
tes = [r[2] for r in l2_results]
norms = [r[3] for r in l2_results]

ax = axes[0]
ax.plot(Cs_arr, trs, marker="o", linewidth=2.5, color="#DC2626", label="학습")
ax.plot(Cs_arr, tes, marker="s", linewidth=2.5, color="#0D9488", label="시험")
best_C = max(l2_results, key=lambda r: r[2])[0]
ax.axvline(best_C, color="#1E40AF", linestyle="--", linewidth=1.5)
ax.set_xscale("log")
ax.set_xlabel("C (작을수록 강한 정규화)")
ax.set_ylabel("정확도")
ax.set_title("정규화 강도와 성능")
ax.legend(fontsize=9)
ax.grid(alpha=0.3, which="both")
ax.invert_xaxis()

ax = axes[1]
ax.plot(Cs_arr, norms, marker="o", linewidth=2.5, color="#EA580C")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("C (작을수록 강한 정규화)")
ax.set_ylabel("가중치 노름 (로그)")
ax.set_title("정규화가 가중치를 줄인다")
ax.grid(alpha=0.3, which="both")
ax.invert_xaxis()

plt.tight_layout()
plt.show()

print("왼쪽 그래프가 이론편 11.3절의 전형적인 모양이다.")
print("  너무 약하면 과대적합, 너무 강하면 과소적합 — 사이에 최적점이 있다")
print()
print("[PyTorch 에서는]")
print("  optimizer = torch.optim.Adam(params, lr=1e-3, weight_decay=1e-4)")
print("  weight_decay 가 위 식의 λ 에 해당한다 (C 와 반대 방향)")
print()
print("  12장에서 다뤘듯 Adam 에는 AdamW 를 쓰는 것이 낫다.")

---

## 3. Dropout ★ — 이론편 11.3절

**학습 중 뉴런 일부를 무작위로 끈다.**

```
학습 시: 각 뉴런을 확률 p 로 0으로 만든다
추론 시: 전부 사용한다
```

**왜 효과가 있나**

특정 뉴런에 과하게 의존하는 것을 막는다. 매번 다른 뉴런이 꺼지므로
**어느 하나가 없어도 동작하도록** 학습된다.

앙상블(9장)과 비슷한 효과라는 해석도 있다 — 매번 다른 부분망을 학습시키는 셈이다.

### 역스케일링 — 왜 (1−p)로 나누나

In [ ]:
import numpy as np

print("=" * 78)
print("Dropout 의 기댓값 보존")
print("=" * 78)
print()
print("문제: 학습 때는 일부를 끄고 추론 때는 다 켜면 출력 크기가 달라진다.")
print()

np.random.seed(42)
activations = np.random.rand(100000) * 2      # 활성값 (양수)

print(f"{'p':<10}{'원본 평균':<16}{'단순 마스킹':<18}{'(1-p)로 나눔':<18}{'보존?'}")
print("-" * 78)

for p in [0.2, 0.5, 0.8]:
    mask = (np.random.rand(len(activations)) > p).astype(float)
    naive = activations * mask
    inverted = activations * mask / (1 - p)

    ok = abs(inverted.mean() - activations.mean()) < 0.02
    print(f"{p:<10}{activations.mean():<16.4f}{naive.mean():<18.4f}"
          f"{inverted.mean():<18.4f}{'예' if ok else '아니오'}")

print("-" * 78)
print()
print("[단순 마스킹의 문제]")
print("  p=0.5 면 평균이 절반이 된다.")
print("  학습 때 절반 크기로 학습했는데 추론 때 전부 켜면 출력이 두 배가 된다.")
print()
print("[해결: 역스케일링 (inverted dropout)]")
print("  학습 때 (1-p) 로 나눠 기댓값을 맞춘다.")
print("  → 추론 때는 아무것도 하지 않아도 된다")
print()
print("  PyTorch 의 nn.Dropout 이 이 방식을 쓴다.")

In [ ]:
import torch
import torch.nn as nn
import numpy as np

print("=" * 70)
print("PyTorch Dropout 동작 확인")
print("=" * 70)

torch.manual_seed(0)
x = torch.ones(1, 10)
dropout = nn.Dropout(p=0.5)

print(f"입력: {x.numpy()[0]}")
print()

print("[학습 모드] — 일부가 0이 되고 나머지는 2배")
dropout.train()
for i in range(3):
    out = dropout(x)
    n_zero = (out == 0).sum().item()
    print(f"  {i+1}회: {out.numpy()[0]}  (0인 개수 {n_zero})")

print()
print("[추론 모드] — 아무것도 하지 않는다")
dropout.eval()
out_eval = dropout(x)
print(f"  {out_eval.numpy()[0]}")
print()

# 기댓값 확인
dropout.train()
torch.manual_seed(0)
samples = torch.stack([dropout(x) for _ in range(5000)])
print(f"학습 모드 5000회 평균: {samples.mean().item():.4f}")
print(f"추론 모드 출력       : {out_eval.mean().item():.4f}")
print(f"차이                : {abs(samples.mean().item() - out_eval.mean().item()):.4f}")
print()
print("[OK] 기댓값이 보존된다")
print()
print("=" * 70)
print("[가장 흔한 실수] model.eval() 을 잊는 것")
print("=" * 70)
print("  평가할 때 train 모드로 두면 뉴런이 무작위로 꺼진 채 예측한다.")
print("  → 결과가 매번 달라지고 성능도 떨어진다")
print()
print("  13장에서 배운 model.train() / model.eval() 전환이 여기서 중요해진다.")

---

## 4. Batch Normalization ★ — 이론편 11.3절

**각 층의 입력 분포를 정규화한다.**

$$\hat{x} = \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}, \qquad y = \gamma\hat{x} + \beta$$

$\mu_B, \sigma_B$는 **미니배치 안에서** 계산한다. 이것이 이름의 유래다.

이론편 11.3절에서 손으로 계산한 값을 확인하자.

In [ ]:
import numpy as np
import torch
import torch.nn as nn

print("=" * 70)
print("BatchNorm 손계산 — 이론편 11.3절 값 검증")
print("=" * 70)

x = np.array([1.0, 2.0, 3.0, 4.0])
eps = 1e-5

mean = x.mean()
var = x.var()
normalized = (x - mean) / np.sqrt(var + eps)

print(f"입력   : {x}")
print(f"평균   : {mean}")
print(f"분산   : {var}")
print(f"표준편차: {np.sqrt(var):.6f}")
print()
print("정규화 계산")
for xi, ni in zip(x, normalized):
    print(f"  ({xi} - {mean}) / sqrt({var} + {eps}) = {ni:.6f}")
print()
print(f"결과   : {normalized.round(6)}")
print(f"평균   : {normalized.mean():.6f}   (0 이어야 함)")
print(f"표준편차: {normalized.std():.6f}   (1 이어야 함)")
print()

# PyTorch 와 대조
bn = nn.BatchNorm1d(1, eps=eps, affine=False)
bn.train()
torch_out = bn(torch.tensor(x, dtype=torch.float32).view(-1, 1)).view(-1).numpy()

print(f"PyTorch: {torch_out.round(6)}")
print("-" * 70)
assert np.allclose(normalized, torch_out, atol=1e-4)
print("[OK] 손계산과 일치 (이론편 11.3절 값: -1.3416, -0.4472, 0.4472, 1.3416)")
print()
print("[gamma 와 beta 의 역할]")
print("  정규화가 항상 좋은 것은 아니다.")
print("  학습 가능한 gamma(크기)와 beta(이동)를 두어")
print("  모델이 필요하면 원래대로 되돌릴 수 있게 한다.")

In [ ]:
import torch
import torch.nn as nn
import numpy as np

print("=" * 78)
print("BatchNorm 의 학습 모드와 추론 모드")
print("=" * 78)
print()
print("학습 시: 현재 배치의 평균·분산 사용")
print("추론 시: 학습 중 누적한 이동평균 사용")
print("  → 배치 크기가 1이어도 동작한다")
print()

torch.manual_seed(0)
bn = nn.BatchNorm1d(4, momentum=0.1)

print(f"{'단계':<10}{'입력 평균':<16}{'누적 평균 (running_mean)'}")
print("-" * 78)

bn.train()
for step in range(1, 6):
    batch = torch.randn(16, 4) * 2 + 5      # 평균 5, 표준편차 2
    _ = bn(batch)
    print(f"{step:<10}{batch.mean().item():<16.4f}"
          f"{bn.running_mean.mean().item():.4f}")

print("-" * 78)
print()
print(f"실제 데이터 평균  : 5.0")
print(f"누적된 running_mean: {bn.running_mean.mean().item():.4f}")
print()
print("momentum=0.1 이면 매 스텝 10%씩 갱신한다.")
print("  running_mean ← 0.9 x running_mean + 0.1 x batch_mean")
print()
print("  12장의 모멘텀과 같은 형태의 이동평균이다.")
print()

# 배치 크기 1 문제
print("=" * 78)
print("[주의] 학습 시 배치 크기가 1이면")
print("=" * 78)
bn.train()
try:
    single = torch.randn(1, 4)
    _ = bn(single)
    print("  통과 (분산이 0이라 불안정할 수 있음)")
except Exception as e:
    print(f"  오류: {type(e).__name__}")
    print(f"  {str(e)[:80]}")
print()
print("  배치가 1개면 분산이 정의되지 않는다.")
print("  → 이것이 18장 Transformer 가 LayerNorm 을 쓰는 이유 중 하나다.")

In [ ]:
import torch
import torch.nn as nn
import numpy as np

print("=" * 78)
print("BatchNorm vs LayerNorm (18장 5절 미리보기)")
print("=" * 78)

X = torch.tensor([[1.0, 2.0, 3.0, 4.0],
                  [10.0, 20.0, 30.0, 40.0]])

print("입력 (샘플 2개, 특성 4개)")
print(X.numpy())
print()

bn = nn.BatchNorm1d(4, affine=False)
bn.train()
# ── nn.LayerNorm 파라미터 ────────────────────────────────────
#   normalized_shape   정규화할 차원.  예: 768 또는 (seq, 768)
#   eps                기본값 1e-5
#   elementwise_affine gamma·beta 사용.  기본값 True
#
#   BatchNorm 과의 차이
#     BatchNorm: 배치 안의 같은 특성끼리 정규화 → 배치 크기 의존
#     LayerNorm: 한 샘플의 모든 특성을 정규화 → 배치와 무관
#   → Transformer 가 LayerNorm 을 쓰는 이유 (시퀀스 길이가 가변)
# ──────────────────────────────────────────────────────────────
ln = nn.LayerNorm(4, elementwise_affine=False)

print("BatchNorm — 각 열(특성)을 정규화")
print(bn(X).detach().numpy().round(3))
print("  → 각 열에서 두 값이 -1, +1 이 되었다")
print()
print("LayerNorm — 각 행(샘플)을 정규화")
print(ln(X).numpy().round(3))
print("  → 두 행이 같은 값이 되었다 (비율이 같으므로)")
print()
print("-" * 78)
print(f"{'':16}{'BatchNorm':<28}{'LayerNorm'}")
print("-" * 78)
print(f"{'정규화 기준':<16}{'배치 안의 같은 특성':<28}{'한 샘플의 모든 특성'}")
print(f"{'배치 크기 의존':<16}{'있음':<28}{'없음'}")
print(f"{'추론 시':<16}{'이동평균 사용':<28}{'그때그때 계산'}")
print(f"{'주로 쓰이는 곳':<16}{'CNN (12장)':<28}{'Transformer (18장)'}")
print("-" * 78)

---

## 5. 실제 학습에서 비교 — 이론편 11.3절

**이제 실제로 효과가 있는지 확인한다.**

솔직히 말하면, **기대만큼 극적이지 않을 수 있다.** 그 이유도 함께 살펴본다.

In [ ]:
import time
import numpy as np

print("=" * 78)
print("정규화 기법 비교 — 150에폭 학습")
print("=" * 78)
print("(각 설정마다 학습하므로 3~5분 걸립니다)")
print()

configs = [
    ("정규화 없음",       {"dropout": 0.0, "batchnorm": False}, 0.0),
    ("Dropout 0.3",     {"dropout": 0.3, "batchnorm": False}, 0.0),
    ("Dropout 0.5",     {"dropout": 0.5, "batchnorm": False}, 0.0),
    ("weight_decay",    {"dropout": 0.0, "batchnorm": False}, 1e-3),
    ("BatchNorm",       {"dropout": 0.0, "batchnorm": True},  0.0),
    ("Dropout + WD",    {"dropout": 0.3, "batchnorm": False}, 1e-3),
]

all_results = {}
t0 = time.time()

for name, model_kw, wd in configs:
    hist = train_model(make_model(**model_kw), epochs=150, weight_decay=wd)
    all_results[name] = hist
    print(f"  {name:<20} 완료 ({time.time()-t0:.0f}초)")

print()
print("=" * 78)
print(f"{'설정':<20}{'최종 학습':<12}{'최종 시험':<12}{'최고 시험':<12}"
      f"{'최저 검증손실':<16}{'최고 시점'}")
print("-" * 78)

for name, hist in all_results.items():
    best_acc = max(hist["test_acc"])
    best_epoch = int(np.argmax(hist["test_acc"])) + 1
    min_loss = min(hist["test_loss"])
    print(f"{name:<20}{hist['train_acc'][-1]:<12.4f}{hist['test_acc'][-1]:<12.4f}"
          f"{best_acc:<12.4f}{min_loss:<16.4f}{best_epoch}")

print("-" * 78)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

colors = ["#DC2626", "#EA580C", "#F59E0B", "#0D9488", "#1E40AF", "#7C3AED"]

# --- 왼쪽: 시험 정확도 ---
ax = axes[0]
for (name, hist), color in zip(all_results.items(), colors):
    ax.plot(hist["test_acc"], linewidth=1.8, color=color, label=name, alpha=0.85)
ax.set_xlabel("에폭")
ax.set_ylabel("시험 정확도")
ax.set_title("시험 정확도 — 차이가 크지 않다")
ax.legend(fontsize=7, loc="lower right")
ax.grid(alpha=0.3)

# --- 오른쪽: 시험 손실 ---
ax = axes[1]
for (name, hist), color in zip(all_results.items(), colors):
    ax.plot(hist["test_loss"], linewidth=1.8, color=color, label=name, alpha=0.85)
ax.set_xlabel("에폭")
ax.set_ylabel("시험 손실")
ax.set_title("시험 손실 — 여기서 차이가 보인다")
ax.legend(fontsize=7)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("=" * 78)
print("정직하게 읽기")
print("=" * 78)
print()

base_acc = max(all_results["정규화 없음"]["test_acc"])
base_loss = min(all_results["정규화 없음"]["test_loss"])

print(f"{'설정':<20}{'최고 정확도 변화':<20}{'최저 손실 변화'}")
print("-" * 78)
for name, hist in all_results.items():
    d_acc = max(hist["test_acc"]) - base_acc
    d_loss = min(hist["test_loss"]) - base_loss
    print(f"{name:<20}{d_acc:>+16.4f}    {d_loss:>+16.4f}")
print("-" * 78)
print()
print("[관찰 1] 정확도 차이는 작다")
print("  이 규모의 실습에서는 정규화의 효과가 정확도로 잘 드러나지 않는다.")
print()
print("[관찰 2] 손실은 명확히 개선된다")
print("  Dropout 을 쓰면 검증 손실이 낮아진다 = 예측이 덜 과신한다.")
print()
print("[관찰 3] 최고 성능 시점이 늦춰진다")
print("  정규화가 없으면 일찍 과대적합에 빠진다.")
print("  정규화를 쓰면 더 오래 학습해도 괜찮다.")

### 왜 효과가 기대만큼 크지 않은가

**정직하게 다루자.** 교과서 그림처럼 극적인 차이가 안 나오는 이유가 있다.

| 이유 | 설명 |
|---|---|
| 데이터가 너무 적음 | 500장으로는 어떤 정규화도 한계 |
| 모델이 이미 외워버림 | 학습 정확도 1.0에서는 정규화가 늦음 |
| 표 형태가 아닌 이미지 | Dropout은 큰 모델·큰 데이터에서 효과 |
| 실습 규모 | 실제 학습은 수백만 장, 수백 에폭 |

**그래도 배울 것이 있다.**

1. **정확도만 보면 안 된다** — 손실을 함께 봐야 과대적합이 보인다
2. **정규화는 만병통치약이 아니다** — 데이터를 늘리는 것이 가장 확실
3. **조기 종료가 가장 효과적일 때가 많다** — 6절에서 다룬다

---

## 6. 조기 종료 — 이론편 11.6절

**가장 단순하고 가장 효과적인 방법.**

검증 성능이 나빠지기 시작하면 멈춘다. 1절의 그래프에서 이미 봤듯,
**최적 시점은 학습 도중에 지나간다.**

In [ ]:
import numpy as np


class EarlyStopping:
    """조기 종료 (이론편 11.6절)

    patience 에폭 동안 개선이 없으면 멈춘다.
    """

    def __init__(self, patience=10, min_delta=0.0, mode="min"):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best = np.inf if mode == "min" else -np.inf
        self.counter = 0
        self.best_epoch = 0
        self.should_stop = False

    def step(self, value, epoch):
        improved = (value < self.best - self.min_delta) if self.mode == "min" \
                   else (value > self.best + self.min_delta)
        if improved:
            self.best = value
            self.best_epoch = epoch
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop


print("=" * 78)
print("조기 종료 시뮬레이션")
print("=" * 78)
print()
print("1절에서 얻은 검증 손실 곡선에 적용해 본다.")
print()

for patience in [5, 10, 20, 30]:
    stopper = EarlyStopping(patience=patience, mode="min")
    stopped_at = None

    for epoch, loss in enumerate(baseline["test_loss"], 1):
        if stopper.step(loss, epoch):
            stopped_at = epoch
            break

    if stopped_at is None:
        stopped_at = len(baseline["test_loss"])

    acc_at_best = baseline["test_acc"][stopper.best_epoch - 1]
    final_acc = baseline["test_acc"][-1]

    print(f"patience={patience:<4}  {stopped_at:>3}에폭에서 중단  "
          f"최적 {stopper.best_epoch:>3}에폭 (손실 {stopper.best:.4f}, "
          f"정확도 {acc_at_best:.4f})")

print()
print(f"끝까지 학습했을 때: 150에폭, 정확도 {baseline['test_acc'][-1]:.4f}, "
      f"손실 {baseline['test_loss'][-1]:.4f}")
print()
print("[조기 종료의 이점]")
print("  1) 학습 시간 절약")
print("  2) 과대적합 방지")
print("  3) 하이퍼파라미터가 patience 하나뿐")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(9, 4.5))

epochs_range = range(1, len(baseline["test_loss"]) + 1)
ax.plot(epochs_range, baseline["train_loss"], linewidth=2,
        color="#DC2626", label="학습 손실")
ax.plot(epochs_range, baseline["test_loss"], linewidth=2,
        color="#0D9488", label="검증 손실")

best_epoch = int(np.argmin(baseline["test_loss"])) + 1
ax.axvline(best_epoch, color="#1E40AF", linewidth=2, linestyle="--")
ax.text(best_epoch + 3, max(baseline["test_loss"]) * 0.85,
        f"최적 지점\n{best_epoch}에폭", fontsize=9, color="#1E40AF")

# patience=10 으로 멈추는 지점
stopper = EarlyStopping(patience=10, mode="min")
stop_at = len(baseline["test_loss"])
for epoch, loss in enumerate(baseline["test_loss"], 1):
    if stopper.step(loss, epoch):
        stop_at = epoch
        break
ax.axvline(stop_at, color="#EA580C", linewidth=2, linestyle=":")
ax.text(stop_at + 3, max(baseline["test_loss"]) * 0.6,
        f"중단\n{stop_at}에폭", fontsize=9, color="#EA580C")

ax.axvspan(best_epoch, len(baseline["test_loss"]), alpha=0.12, color="#DC2626")
ax.set_xlabel("에폭")
ax.set_ylabel("손실")
ax.set_title("조기 종료 — 빨간 영역은 과대적합 구간")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("=" * 78)
print("실무에서의 조기 종료")
print("=" * 78)
print()
print("보통 '최적 시점의 가중치'를 따로 저장한다.")
print()
code_example = [
    "best_loss = float('inf')",
    "patience_counter = 0",
    "",
    "for epoch in range(max_epochs):",
    "    train_one_epoch(model, train_loader)",
    "    val_loss = evaluate(model, val_loader)",
    "",
    "    if val_loss < best_loss:",
    "        best_loss = val_loss",
    "        torch.save(model.state_dict(), 'best.pt')   # 최적 시점 저장",
    "        patience_counter = 0",
    "    else:",
    "        patience_counter += 1",
    "        if patience_counter >= patience:",
    "            break",
    "",
    "model.load_state_dict(torch.load('best.pt'))        # 최적 시점 복원",
]
for line in code_example:
    print("  " + line)
print()
print("[주의] 시험 데이터로 조기 종료하면 안 된다")
print("  그것은 시험 데이터를 학습에 쓰는 것이다 (6장 7절의 데이터 누수).")
print("  학습 / **검증** / 시험 세 개로 나눠야 한다.")

---

## 7. 데이터 증강 — 이론편 11.4절

**가장 확실한 방법은 데이터를 늘리는 것이다.**

새 데이터를 구하기 어려우면, **있는 데이터를 변형해** 늘린다.

In [ ]:
import torch
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np

print("=" * 70)
print("이미지 증강 기법")
print("=" * 70)

sample_img = train_data[0][0]

augmentations = {
    "원본": lambda x: x,
    "좌우 반전": transforms.RandomHorizontalFlip(p=1.0),
    "회전 15도": transforms.RandomRotation(15),
    "이동": transforms.RandomAffine(0, translate=(0.15, 0.15)),
    "확대/축소": transforms.RandomAffine(0, scale=(0.8, 1.2)),
    "지우기": transforms.RandomErasing(p=1.0, scale=(0.05, 0.15)),
}

fig, axes = plt.subplots(1, len(augmentations), figsize=(14, 2.6))
torch.manual_seed(3)

for ax, (name, aug) in zip(axes, augmentations.items()):
    img = aug(sample_img)
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(name, fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

print()
print("[증강의 원리]")
print("  '옷을 좌우로 뒤집어도 여전히 그 옷이다'")
print("  이 지식을 모델에게 데이터로 알려주는 것이다.")
print()
print("[주의] 아무 변형이나 되는 것은 아니다")
print(f"{'변형':<20}{'FashionMNIST':<20}{'숫자 인식(MNIST)'}")
print("-" * 70)
print(f"{'좌우 반전':<20}{'적절':<20}{'부적절 (2와 5가 헷갈림)'}")
print(f"{'상하 반전':<20}{'부적절':<20}{'부적절'}")
print(f"{'큰 회전':<20}{'부적절':<20}{'부적절 (6과 9)'}")
print("-" * 70)
print()
print("  → 도메인 지식이 필요하다. 무엇이 '같은 것'인지 알아야 한다.")

In [ ]:
print("=" * 78)
print("다른 영역의 데이터 증강")
print("=" * 78)
print()
print(f"{'영역':<16}{'기법':<38}{'주의점'}")
print("-" * 78)
techniques = [
    ("이미지", "반전·회전·자르기·색조 변경", "의미가 바뀌지 않아야"),
    ("텍스트", "동의어 교체·역번역·문장 순서", "의미 왜곡 위험"),
    ("음성", "속도 변경·노이즈 추가·피치", "화자 특성 변화 주의"),
    ("표 데이터", "SMOTE (소수 클래스 합성)", "비현실적 표본 생성 가능"),
    ("공통", "Mixup (두 표본을 섞기)", "라벨도 함께 섞어야"),
]
for a, b, c in techniques:
    print(f"{a:<16}{b:<38}{c}")
print("-" * 78)
print()
print("[LLM 시대의 데이터 증강]")
print()
print("  LLM 으로 학습 데이터를 생성하는 방법이 널리 쓰인다.")
print("  '이 문장을 다르게 표현해줘' 를 반복해 데이터를 늘리는 식이다.")
print()
print("  31장 SFT 실습에서 데이터가 부족했던 것을 떠올려 보자.")
print("  실무에서는 이런 방식으로 수천 건을 만들기도 한다.")
print()
print("[가장 확실한 것]")
print("  증강은 어디까지나 보조 수단이다.")
print("  **진짜 새로운 데이터**를 구하는 것이 언제나 가장 낫다.")

---

## 8. 무엇을 언제 쓸 것인가 — 이론편 11.3절

In [ ]:
print("=" * 78)
print("정규화 기법 선택 가이드")
print("=" * 78)
print()
print(f"{'기법':<20}{'효과':<26}{'대가':<22}{'권장 시점'}")
print("-" * 78)
methods = [
    ("데이터 늘리기", "가장 확실", "비용·시간", "언제나 1순위"),
    ("조기 종료", "간단·효과적", "거의 없음", "**항상 사용**"),
    ("weight_decay", "가중치 억제", "λ 조정 필요", "기본으로 켜기"),
    ("Dropout", "공적응 방지", "학습이 느려짐", "큰 모델·FC 층"),
    ("BatchNorm", "학습 안정화", "배치 의존", "CNN"),
    ("LayerNorm", "학습 안정화", "—", "Transformer"),
    ("데이터 증강", "다양성 확보", "도메인 지식 필요", "이미지·음성"),
    ("모델 축소", "근본 해결", "표현력 감소", "데이터가 아주 적을 때"),
]
for a, b, c, d in methods:
    print(f"{a:<20}{b:<26}{c:<22}{d}")
print("-" * 78)
print()
print("[순서대로 시도하기]")
print()
print("  1. 조기 종료를 켠다             — 비용이 없다")
print("  2. weight_decay 를 넣는다       — 1e-4 ~ 1e-2")
print("  3. 데이터를 늘리거나 증강한다     — 가장 효과적")
print("  4. Dropout 을 추가한다          — 0.1 ~ 0.5")
print("  5. 그래도 안 되면 모델을 줄인다   — 근본 대책")
print()
print("[하지 말아야 할 것]")
print("  처음부터 모든 정규화를 다 켜는 것")
print("  → 무엇이 효과가 있었는지 알 수 없다")
print("  → 하나씩 추가하며 측정한다 (6장 3절의 방식)")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("=" * 78)
print("과대적합 진단 체크리스트")
print("=" * 78)
print()

train_final = baseline["train_acc"][-1]
test_final = baseline["test_acc"][-1]
gap = train_final - test_final

print(f"{'증상':<34}{'해석':<24}{'대책'}")
print("-" * 78)
symptoms = [
    ("학습↑ 검증↑", "정상 학습 중", "계속"),
    ("학습↑ 검증 정체", "과대적합 시작", "조기 종료 준비"),
    ("학습↑ 검증↓", "**과대적합**", "정규화·데이터 추가"),
    ("학습도 낮음", "과소적합", "모델을 키우거나 더 학습"),
    ("둘 다 요동", "학습률 문제", "학습률 낮추기 (12장)"),
]
for a, b, c in symptoms:
    print(f"{a:<34}{b:<24}{c}")
print("-" * 78)
print()
print("현재 실습의 진단")
print(f"  학습 정확도 {train_final:.4f} / 시험 정확도 {test_final:.4f}")
print(f"  격차 {gap:+.4f}")
if gap > 0.15:
    print("  → 명확한 과대적합. 정규화와 데이터 추가가 필요하다.")
elif gap > 0.05:
    print("  → 약한 과대적합.")
else:
    print("  → 격차가 작다.")
print()
print("[격차의 기준]")
print("  절대적인 수치는 없다. 문제와 데이터에 따라 다르다.")
print("  중요한 것은 **검증 손실이 오르기 시작하는 시점**이다.")

---

## 9. 정리

### 확인한 이론편 값

| 이론편 절 | 내용 | 결과 |
|---|---|---|
| **11.3** | **BatchNorm [1,2,3,4] → ±1.3416, ±0.4472** | **일치** ✓ |
| 11.3 | Dropout 역스케일링으로 기댓값 보존 | 검증 ✓ |
| 11.3 | L2가 가중치 노름을 줄임 | 측정 ✓ |
| 11.6 | 조기 종료 지점 | 시뮬레이션 ✓ |

### 세 가지 정규화 비교

| 기법 | 무엇을 하나 | 어디에 넣나 |
|---|---|---|
| weight_decay | 큰 가중치에 벌점 | 옵티마이저 인자 |
| Dropout | 뉴런을 무작위로 끔 | 활성화 함수 뒤 |
| BatchNorm | 층 입력을 정규화 | 선형층 뒤, 활성화 앞 |

### 기억할 것

| 항목 | 요점 |
|---|---|
| **정확도보다 손실** | 과대적합이 손실에서 먼저 보인다 |
| Dropout 역스케일링 | 학습 시 (1−p)로 나눔 → 추론 시 그대로 |
| `model.eval()` | 잊으면 Dropout·BN이 학습 모드로 동작 |
| BatchNorm | 배치 크기 1이면 문제 → Transformer는 LayerNorm |
| **조기 종료** | 가장 간단하고 효과적 |
| 조기 종료 기준 | **검증** 데이터로 (시험 데이터 아님) |
| 데이터 증강 | 도메인 지식 필요 — 무엇이 '같은 것'인가 |
| 순서 | 조기 종료 → weight_decay → 데이터 → Dropout |

### 이 장에서 배운 태도

**정규화가 항상 극적인 효과를 내지는 않는다.**

5절에서 확인했듯 이 규모의 실습에서는 정확도 차이가 작았다.
하지만 **검증 손실은 명확히 개선**되었고, **최적 시점이 늦춰졌다.**

무엇을 기준으로 보느냐에 따라 결론이 달라진다는 것 —
이것이 실험을 읽는 법이다.

### 다음 장

**15. CNN 실습 — 이미지 분류** — 이 장에서 쓴 MLP 대신 합성곱을 쓴다.
BatchNorm과 데이터 증강이 CNN에서 진가를 발휘한다.